In [ ]:
# ── CELL 1: The Two Core Data Structures ──────────────────────────────────────
import pandas as pd
import numpy as np

# Series — 1D labeled array
s = pd.Series([10, 20, 30, 40], index=['a', 'b', 'c', 'd'])
print(s)
print(s['b'])
print(s.dtype)

# DataFrame — 2D labeled table
df = pd.DataFrame({
    'name':   ['Alice', 'Bob', 'Carol', 'Dave'],
    'age':    [25, 30, 35, 28],
    'salary': [70000, 85000, 90000, 75000],
    'dept':   ['Eng', 'Sales', 'Eng', 'HR']
})
print(df)
print(df.shape)
print(df.dtypes)
print(df.columns)

In [ ]:
# ── CELL 2: First Look — Always Do This First ─────────────────────────────────
print(df.head())
print(df.tail())
print(df.shape)
print(df.dtypes)
df.info()
print(df.describe())
print(df.describe(include='all'))
print(df.isnull().sum())
print(df.isnull().sum() / len(df))
print(df.duplicated().sum())
print(df.nunique())
print(df['dept'].value_counts())

In [ ]:
# ── CELL 3: Selecting Data — loc, iloc ────────────────────────────────────────
print(df['age'])
print(df[['name', 'salary']])

# loc — label-based
print(df.loc[0, 'name'])
print(df.loc[0:2, 'name':'age'])
print(df.loc[df['age'] > 28])

# iloc — position-based
print(df.iloc[0])
print(df.iloc[0, 1])
print(df.iloc[0:3, 0:2])
print(df.iloc[:, -1])

# at / iat — single cell
print(df.at[0, 'name'])
print(df.iat[0, 0])

In [ ]:
# ── CELL 4: Filtering — Boolean Conditions ────────────────────────────────────
print(df[df['age'] > 28])
print(df[df['dept'] == 'Eng'])

# Multiple conditions
print(df[(df['age'] > 25) & (df['dept'] == 'Eng')])
print(df[(df['dept'] == 'Eng') | (df['dept'] == 'Sales')])
print(df[~(df['dept'] == 'HR')])

# isin
print(df[df['dept'].isin(['Eng', 'Sales'])])

# between
print(df[df['salary'].between(70000, 85000)])

# String filters
print(df[df['name'].str.startswith('A')])
print(df[df['name'].str.contains('li', case=False)])

# query()
print(df.query("age > 28 and dept == 'Eng'"))
threshold = 80000
print(df.query("salary > @threshold"))

In [ ]:
# ── CELL 5: Adding, Modifying, Dropping Columns ───────────────────────────────
df['bonus'] = df['salary'] * 0.10
df['seniority'] = df['age'].apply(lambda x: 'senior' if x >= 30 else 'junior')
print(df)

df['salary'] = df['salary'] + 5000

df.rename(columns={'dept': 'department'}, inplace=True)

df.drop(columns=['bonus'], inplace=True)

df = df.assign(
    tax=df['salary'] * 0.20,
    net=lambda x: x['salary'] - x['tax']
)
print(df)

In [ ]:
# ── CELL 6: Handling Missing Data ─────────────────────────────────────────────
df_missing = pd.DataFrame({
    'name':   ['Alice', 'Bob', 'Carol', 'Dave'],
    'salary': [70000, np.nan, 90000, np.nan],
    'dept':   ['Eng', 'Sales', None, 'HR']
})

print(df_missing.isnull())
print(df_missing.isnull().sum())

# Drop
print(df_missing.dropna())
print(df_missing.dropna(how='all'))
print(df_missing.dropna(subset=['salary']))

# Fill
print(df_missing.fillna(0))
df_missing['salary'] = df_missing['salary'].fillna(df_missing['salary'].mean())
df_missing['dept'] = df_missing['dept'].fillna('Unknown')
print(df_missing)

# Replace
df_rep = pd.DataFrame({'val': [-999, 10, -999, 30]})
print(df_rep.replace(-999, np.nan))

In [ ]:
# ── CELL 7: apply(), map() ────────────────────────────────────────────────────
df2 = pd.DataFrame({
    'name':   ['Alice', 'Bob', 'Carol', 'Dave'],
    'age':    [25, 30, 35, 28],
    'salary': [70000, 85000, 90000, 75000],
    'dept':   ['Eng', 'Sales', 'Eng', 'HR']
})

# apply on Series
print(df2['salary'].apply(lambda x: x * 1.1))
print(df2['name'].apply(str.upper))
print(df2['age'].apply(lambda x: 'senior' if x >= 30 else 'junior'))

# apply on DataFrame rows
print(df2.apply(lambda row: row['salary'] / row['age'], axis=1))

# map — label encoding with dict
print(df2['dept'].map({'Eng': 1, 'Sales': 2, 'HR': 3}))

In [ ]:
# ── CELL 8: GroupBy ───────────────────────────────────────────────────────────
df3 = pd.DataFrame({
    'name':      ['Alice', 'Bob', 'Carol', 'Dave', 'Eve'],
    'dept':      ['Eng', 'Sales', 'Eng', 'HR', 'Sales'],
    'seniority': ['senior', 'junior', 'junior', 'senior', 'senior'],
    'salary':    [90000, 70000, 75000, 80000, 85000]
})

print(df3.groupby('dept')['salary'].mean())
print(df3.groupby('dept')['salary'].agg(['mean', 'min', 'max', 'count']))
print(df3.groupby(['dept', 'seniority'])['salary'].mean())

# agg with different functions per column
print(df3.groupby('dept').agg({'salary': ['mean', 'max']}))

# transform — add dept avg salary as a new column
df3['dept_avg_salary'] = df3.groupby('dept')['salary'].transform('mean')
print(df3)

# filter — keep departments with avg salary > 78000
print(df3.groupby('dept').filter(lambda g: g['salary'].mean() > 78000))

In [ ]:
# ── CELL 9: Merge and Join ────────────────────────────────────────────────────
employees = pd.DataFrame({
    'emp_id':  [1, 2, 3, 4],
    'name':    ['Alice', 'Bob', 'Carol', 'Dave'],
    'dept_id': [10, 20, 10, 30]
})
departments = pd.DataFrame({
    'dept_id':   [10, 20, 30],
    'dept_name': ['Engineering', 'Sales', 'HR']
})

print(pd.merge(employees, departments, on='dept_id', how='inner'))
print(pd.merge(employees, departments, on='dept_id', how='left'))
print(pd.merge(employees, departments, on='dept_id', how='outer'))

# concat — stack DataFrames
df_a = pd.DataFrame({'x': [1, 2], 'y': [3, 4]})
df_b = pd.DataFrame({'x': [5, 6], 'y': [7, 8]})
print(pd.concat([df_a, df_b], axis=0, ignore_index=True))
print(pd.concat([df_a, df_b], axis=1))

In [ ]:
# ── CELL 10: Sorting and Ranking ──────────────────────────────────────────────
df4 = pd.DataFrame({
    'name':   ['Alice', 'Bob', 'Carol', 'Dave', 'Eve'],
    'dept':   ['Eng', 'Sales', 'Eng', 'HR', 'Sales'],
    'salary': [90000, 70000, 75000, 80000, 85000]
})

print(df4.sort_values('salary', ascending=False))
print(df4.sort_values(['dept', 'salary'], ascending=[True, False]))
df4['salary_rank'] = df4['salary'].rank(ascending=False, method='dense')
print(df4)
print(df4.nlargest(3, 'salary'))
print(df4.nsmallest(2, 'salary'))

In [ ]:
# ── CELL 11: Pivot Tables ─────────────────────────────────────────────────────
df5 = pd.DataFrame({
    'dept':      ['Eng', 'Eng', 'Sales', 'Sales', 'HR'],
    'seniority': ['senior', 'junior', 'senior', 'junior', 'senior'],
    'salary':    [90000, 70000, 85000, 65000, 80000]
})

pivot = df5.pivot_table(
    values='salary',
    index='dept',
    columns='seniority',
    aggfunc='mean',
    fill_value=0
)
print(pivot)

# melt — wide to long
df_wide = pd.DataFrame({
    'name':   ['Alice', 'Bob'],
    'salary': [90000, 70000],
    'bonus':  [9000, 7000]
})
df_long = pd.melt(df_wide, id_vars=['name'], value_vars=['salary', 'bonus'],
                  var_name='metric', value_name='amount')
print(df_long)

In [ ]:
# ── CELL 12: String Operations ────────────────────────────────────────────────
df6 = pd.DataFrame({
    'name':  ['  Alice Smith ', 'bob jones', 'CAROL LEE'],
    'email': ['alice@eng.com', 'bob@sales.com', 'carol@hr.com']
})

print(df6['name'].str.strip())
print(df6['name'].str.lower())
print(df6['name'].str.upper())
print(df6['name'].str.replace(' ', '_'))
print(df6['name'].str.split(' '))
print(df6['name'].str.strip().str.split(' ').str[0])  # first name
print(df6['name'].str.len())
print(df6['name'].str.contains('ali', case=False))
print(df6['email'].str.split('@').str[1])  # domain

In [ ]:
# ── CELL 13: DateTime Operations ──────────────────────────────────────────────
df7 = pd.DataFrame({
    'date_str': ['2024-01-15', '2024-03-22', '2024-07-04', '2024-11-30'],
    'sales':    [1000, 1500, 2000, 1800]
})

df7['date'] = pd.to_datetime(df7['date_str'])
print(df7['date'].dt.year)
print(df7['date'].dt.month)
print(df7['date'].dt.day)
print(df7['date'].dt.dayofweek)
print(df7['date'].dt.quarter)

# Date arithmetic
df7['days_since'] = (pd.Timestamp.now() - df7['date']).dt.days
print(df7['days_since'])

# Resample — monthly total
monthly = df7.set_index('date')['sales'].resample('ME').sum()
print(monthly)

In [ ]:
# ── CELL 14: Performance Tips ─────────────────────────────────────────────────
df8 = pd.DataFrame({
    'age':    np.random.randint(20, 60, 10000),
    'salary': np.random.randint(50000, 150000, 10000),
    'dept':   np.random.choice(['Eng', 'Sales', 'HR', 'Product'], 10000)
})

# Memory before
print("Before:", df8.memory_usage(deep=True).sum() / 1024, "KB")

# Downcast numerics
df8['age'] = pd.to_numeric(df8['age'], downcast='integer')
df8['salary'] = pd.to_numeric(df8['salary'], downcast='integer')

# Convert string to category
df8['dept'] = df8['dept'].astype('category')

# Memory after
print("After: ", df8.memory_usage(deep=True).sum() / 1024, "KB")

# query() — faster filtering on large data
print(df8.query("salary > 120000 and dept == 'Eng'").shape)

In [ ]:
# ── CELL 15: Pandas ↔ NumPy ↔ scikit-learn Bridge ────────────────────────────
from sklearn.preprocessing import StandardScaler

df9 = pd.DataFrame({
    'age':    [25, 30, 35, 28],
    'salary': [70000, 85000, 90000, 75000]
})

# Pandas → NumPy
X = df9[['age', 'salary']].to_numpy()
print("NumPy array:", X)
print("dtype:", X.dtype)

# Pandas → scikit-learn (accepts DataFrame directly)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(df9[['age', 'salary']])
print("Scaled:", X_scaled)

# NumPy results → Pandas
df9['scaled_salary'] = X_scaled[:, 1]
print(df9)

In [ ]:
# ── CELL 16: Practice Exercise ────────────────────────────────────────────────
import pandas as pd
import numpy as np

np.random.seed(42)

df = pd.DataFrame({
    'employee_id': range(1, 101),
    'department':  np.random.choice(['Eng', 'Sales', 'HR', 'Product'], 100),
    'age':         np.random.randint(22, 55, 100),
    'salary':      np.random.randint(50000, 120000, 100).astype(float),
    'years_exp':   np.random.randint(1, 20, 100),
    'score':       np.random.uniform(60, 100, 100).round(1)
})

# Introduce 5% missing values in salary
df.loc[df.sample(frac=0.05, random_state=42).index, 'salary'] = np.nan

# 1. Inspection
print("Shape:", df.shape)
print("\nMissing values:\n", df.isnull().sum())
print("\nDescribe:\n", df.describe())

# 2. Fill missing salary with department median
df['salary'] = df.groupby('department')['salary'].transform(
    lambda x: x.fillna(x.median())
)

# 3. Feature engineering
df['salary_per_year'] = df['salary'] / df['years_exp']
df['seniority'] = df['years_exp'].apply(lambda x: 'senior' if x >= 10 else 'junior')

# 4. GroupBy analysis
summary = df.groupby('department').agg(
    avg_salary=('salary', 'mean'),
    avg_score=('score', 'mean'),
    headcount=('employee_id', 'count')
).round(2)
print("\nDepartment Summary:\n", summary)

# 5. Filter top performers
top = df[(df['score'] > 90) & (df['salary'] > 90000)]
print(f"\nTop performers: {len(top)}")

# 6. Ready for ML
X = df[['age', 'salary', 'years_exp', 'score']].to_numpy()
print("\nFeature matrix shape:", X.shape)
print("dtype:", X.dtype)